# pyaxon na GPU do Colab (CUDA *profundo*)

Este notebook **compila o pyaxon inteiro com CUDA** e mostra que o `matmul` da biblioteca
passa a rodar na GPU de forma **transparente** — sem mudar nenhuma linha de Python. Como o
`ops::matmul` é chamado no forward **e** no backward de toda camada (Linear, attenção, ...),
**o treino inteiro do modelo passa a rodar na GPU** só ligando o CUDA no build.

O que o notebook faz:
1. compila o `_axon.so` com `-DAXON_ENABLE_CUDA=ON -DAXON_USE_CUBLAS=ON`;
2. confirma `ax.cuda_available()` e imprime o nome da GPU;
3. **verifica** que o resultado na GPU bate com a CPU;
4. **mede o speedup** do matmul (GPU vs CPU);
5. **treina o mesmo modelo na CPU e na GPU** e compara tempo + curva de loss.

**Antes de rodar:** `Ambiente de execução -> Alterar tipo de ambiente -> GPU (T4)`.

In [ ]:
!nvidia-smi -L
!nvcc --version

## 1. Clonar o repo e instalar dependências

O repo já está no GitHub — basta rodar. Troque `REPO_URL` só se for um fork seu.

In [ ]:
REPO_URL = "https://github.com/geraldogrise/axon-llm.git"

import os
if not os.path.isdir("axon-llm"):
    !git clone --depth 1 $REPO_URL axon-llm
%cd axon-llm
!pip -q install pybind11 numpy

## 2. Compilar o pyaxon com CUDA + cuBLAS

Uma linha de configure liga o backend de GPU. O `_axon.so` sai direto dentro de
`python/pyaxon/`, então `import pyaxon` já encontra a extensão com GPU.

In [ ]:
import pybind11, subprocess, os
pybind_dir = pybind11.get_cmake_dir()
os.makedirs("build-cuda", exist_ok=True)

cfg = [
    "cmake", "-S", ".", "-B", "build-cuda", "-G", "Ninja",
    "-DCMAKE_BUILD_TYPE=Release",
    "-DAXON_ENABLE_CUDA=ON", "-DAXON_USE_CUBLAS=ON",
    "-DAXON_BUILD_PYTHON=ON", "-DAXON_BUILD_TESTS=OFF", "-DAXON_BUILD_EXAMPLES=OFF",
    "-DAXON_ENABLE_NATIVE=OFF",  # Colab CPU != build CPU; keep it portable
    f"-Dpybind11_DIR={pybind_dir}",
]
!apt-get -qq install -y ninja-build > /dev/null
print(subprocess.run(cfg, capture_output=True, text=True).stdout[-1500:])
print(subprocess.run(["cmake", "--build", "build-cuda", "-j"], capture_output=True, text=True).stdout[-1500:])
!ls -la python/pyaxon/_axon*.so

## 3. Importar e confirmar a GPU

In [ ]:
import sys; sys.path.insert(0, "python")
import pyaxon as ax, numpy as np

print("cuda_available :", ax.cuda_available())
print("device         :", ax.cuda_device_name())
print("cuda_enabled   :", ax.is_cuda_enabled())
print("matmul thresh  :", ax.cuda_matmul_threshold(), "(m*k*n acima disso vai pra GPU)")
assert ax.cuda_available(), "Compilou sem CUDA — confira o log do configure acima."

## 4. Correção: GPU bate com a CPU?

Mesmo `ax.matmul`; só ligamos/desligamos o dispatch pra GPU e comparamos.

In [ ]:
rng = np.random.default_rng(0)
A = rng.standard_normal((512, 512)).astype(np.float32)
B = rng.standard_normal((512, 512)).astype(np.float32)
ta, tb = ax.from_numpy(A), ax.from_numpy(B)

ax.set_cuda_matmul_threshold(0)          # força TODO matmul pra GPU
ax.set_cuda_enabled(False); cpu = ax.matmul(ta, tb).numpy()
ax.set_cuda_enabled(True);  gpu = ax.matmul(ta, tb).numpy()
ref = A @ B
print("erro GPU vs NumPy:", np.abs(gpu - ref).max())
print("erro GPU vs CPU  :", np.abs(gpu - cpu).max())
assert np.abs(gpu - ref).max() < 1e-2, "resultado da GPU divergiu"

## 5. Benchmark do matmul (GPU vs CPU)

In [ ]:
import time

def bench(n, iters=20):
    A = rng.standard_normal((n, n)).astype(np.float32)
    B = rng.standard_normal((n, n)).astype(np.float32)
    ta, tb = ax.from_numpy(A), ax.from_numpy(B)
    def run():
        for _ in range(iters): ax.matmul(ta, tb)
    ax.set_cuda_enabled(False); run(); t0 = time.perf_counter(); run(); cpu = (time.perf_counter()-t0)/iters
    ax.set_cuda_enabled(True);  run(); t0 = time.perf_counter(); run(); gpu = (time.perf_counter()-t0)/iters
    gflop = 2 * n**3 / 1e9
    print(f"{n:>5}³  CPU {cpu*1e3:8.2f} ms ({gflop/cpu:7.1f} GFLOPS)   "
          f"GPU {gpu*1e3:8.2f} ms ({gflop/gpu:8.1f} GFLOPS)   speedup {cpu/gpu:6.1f}x")

ax.set_cuda_matmul_threshold(0)
for n in (256, 512, 1024, 2048):
    bench(n)

## 6. O ponto principal: **treinar** na GPU sem mudar o código

O mesmo loop de treino (`loss.backward()` + `opt.step()`) roda na CPU e na GPU. Como o
matmul do forward **e** do backward despacha pra GPU, o treino inteiro acelera. Conferimos
que a curva de loss é praticamente idêntica (o backend não muda a matemática).

In [ ]:
def make_model_data(seed=0):
    m = ax.nn.Sequential([
        ax.nn.Linear(1024, 1024, seed=1), ax.nn.ReLU(),
        ax.nn.Linear(1024, 1024, seed=2), ax.nn.ReLU(),
        ax.nn.Linear(1024, 1024, seed=3),
    ])
    r = np.random.default_rng(seed)
    X = ax.from_numpy(r.standard_normal((512, 1024)).astype(np.float32))
    Y = ax.from_numpy(r.standard_normal((512, 1024)).astype(np.float32))
    return m, X, Y

def train(use_gpu, steps=30):
    ax.set_cuda_enabled(use_gpu); ax.set_cuda_matmul_threshold(0)
    m, X, Y = make_model_data()
    opt = ax.optim.Adam(m.parameters(), lr=1e-3)
    losses = []
    m(X); ax.matmul(X, X)  # warmup (JIT cublas handle / first cudaMalloc)
    t0 = time.perf_counter()
    for _ in range(steps):
        loss = ax.mse_loss(m(X), Y)
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
    return (time.perf_counter() - t0) / steps, losses

cpu_t, cpu_l = train(False)
gpu_t, gpu_l = train(True)
print(f"CPU: {cpu_t*1e3:7.1f} ms/step   loss {cpu_l[0]:.4f} -> {cpu_l[-1]:.4f}")
print(f"GPU: {gpu_t*1e3:7.1f} ms/step   loss {gpu_l[0]:.4f} -> {gpu_l[-1]:.4f}")
print(f"speedup no treino: {cpu_t/gpu_t:.1f}x")
print(f"diff máx da curva de loss (CPU vs GPU): {max(abs(a-b) for a,b in zip(cpu_l, gpu_l)):.2e}")

## O que isto prova

- **Correção:** o matmul na GPU bate com a CPU/NumPy (erro ~1e-3, só ruído de float32).
- **Speedup:** matmuls grandes rodam muito mais rápido na GPU (cuBLAS no T4).
- **Profundo:** ligar o CUDA acelera o **treino inteiro** — mesmo código Python, mesma
  curva de loss — porque o forward e o backward de cada camada passam pelo `ops::matmul`,
  que despacha pra GPU acima do `cuda_matmul_threshold`.

### Controles expostos (todos em `pyaxon`)
| Função | O que faz |
|---|---|
| `ax.cuda_available()` | compilado com CUDA? |
| `ax.cuda_device_name()` | nome da GPU ativa |
| `ax.set_cuda_enabled(bool)` | liga/desliga o dispatch pra GPU |
| `ax.set_cuda_matmul_threshold(work)` | `m*k*n` mínimo pra ir pra GPU (crossover do round-trip) |

### Limitação honesta / próximo passo
Cada matmul faz o round-trip host→device→host (copia as matrizes pra GPU e o resultado de
volta). Isso já compensa em matrizes grandes, mas para o máximo desempenho o próximo passo
é o **Tensor viver na GPU** (`Device::CUDA`) — os pesos e ativações ficam residentes e as
cópias somem. É um trabalho maior, mas a base (dispatch por device, kernels tiled/cuBLAS,
toggle em runtime) já está aqui.